# 🎯 Ćwiczenie 8: MLlib — Predykcja CTR reklam + Wykrywanie fraudów (tuning hiperparametrów)

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 8 (Studia stacjonarne, Semestr Zimowy 2026/2027)
**Szacowany czas:** ⏱️ ~180 minut (2× ~90 minut)
**Poziom trudności:** ⭐⭐⭐⭐–⭐⭐⭐⭐⭐ (Zaawansowany)

To laboratorium łączy dwa powiązane tematy ML w Spark MLlib w jednym notebooku:

- **Część A — Predykcja CTR reklam** — pełny `Pipeline` MLlib (AdBoost 📱🖱️)
- **Część B — Wykrywanie fraudów** — tuning hiperparametrów `CrossValidator` (CardShield 💳🕵️)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:

**Część A — Predykcja CTR reklam:**
1. Przygotować zmienne kategoryczne pod ML za pomocą `StringIndexer` i `OneHotEncoder`
2. Zbudować pełny `Pipeline` MLlib łączący preprocessing i model
3. Wytrenować i porównać `LogisticRegression` oraz `RandomForestClassifier`
4. Ewaluować modele przy pomocy `BinaryClassificationEvaluator` (AUC) i `MulticlassClassificationEvaluator` (accuracy, precision, recall)
5. Zapisać i wczytać wytrenowany `PipelineModel`

**Część B — Wykrywanie fraudów:**
6. Tuningować hiperparametry modelu MLlib przy pomocy `ParamGridBuilder` i `CrossValidator`
7. Wybrać właściwą metrykę ewaluacji dla mocno niezbalansowanych danych (AUC-PR vs AUC-ROC)
8. Zastosować ważenie klas (`weightCol`) jako sposób radzenia sobie z class imbalance w Spark
9. Wytrenować i ocenić `GBTClassifier` na dużym zbiorze danych
10. Wyciągnąć najlepsze hiperparametry z `CrossValidatorModel` i policzyć macierz konfuzji przy niestandardowym progu

## 📋 Wymagania wstępne
- Znajomość PySpark DataFrame (Wykłady 4-5)
- Znajomość scikit-learn: Pipeline, klasyfikacja, ewaluacja (Wykład 7)

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz, przeczytaj komórki z podpowiedziami (💡 Hint) lub zapytaj prowadzącego!
> Notebook ma dwie niezależne części — każda generuje własny zbiór danych i może być uruchamiana samodzielnie. Część B (`CrossValidator` + `GBTClassifier`) może potrwać kilka minut.


---

# 🅰️ Część A: MLlib — Predykcja kliknięć w reklamy (CTR) 📱🖱️

Pracujesz jako data scientist w agencji reklamowej **AdBoost**. Masz ~300 000 wyświetleń reklam — zbyt dużo, by wygodnie iterować w scikit-learn, czas na Spark MLlib!


In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

import random
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

random.seed(42)
np.random.seed(42)

spark = (
    SparkSession.builder
    .appName("AdBoost-CTR-Prediction")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("✅ Setup complete! SparkSession gotowa 🚀")
print(f"Wersja Sparka: {spark.version}")

## 📖 Kontekst: AdBoost 📱🖱️

Pracujesz jako data scientist w agencji reklamowej **AdBoost**. Zespół sprzedaży chce wiedzieć, które reklamy mają największe szanse na kliknięcie (CTR — *Click-Through Rate*) — zanim jeszcze zostaną wyświetlone. Masz ~300 000 wyświetleń reklam z ostatniego miesiąca. To zdecydowanie za dużo, by wygodnie iterować w scikit-learn na laptopie — czas na **Spark MLlib**!

### Kluczowe koncepty MLlib

| Koncept | Opis | Przykład |
|---------|------|----------|
| `StringIndexer` | koduje string → indeks liczbowy | `StringIndexer(inputCol="device_type", outputCol="device_type_idx")` |
| `OneHotEncoder` | koduje indeks → wektor rzadki (bez fałszywej kolejności) | `OneHotEncoder(inputCol="device_type_idx", outputCol="device_type_ohe")` |
| `VectorAssembler` | łączy kolumny w jedną kolumnę wektorową `features` | `VectorAssembler(inputCols=[...], outputCol="features")` |
| `Pipeline` | łańcuch Transformerów/Estimatorów, `fit()` → `PipelineModel` | `Pipeline(stages=[...])` |
| `BinaryClassificationEvaluator` | mierzy AUC dla klasyfikacji binarnej | `evaluator.evaluate(predictions)` |

In [ ]:
# @title 📊 Generowanie zbioru danych CTR (~300 000 wyświetleń)

N = 300_000

ad_categories = ["moda", "elektronika", "podroze", "jedzenie", "sport", "finanse", "uroda", "gry"]
times_of_day = ["rano", "poludnie", "popoludnie", "wieczor", "noc"]
device_types = ["mobile", "desktop", "tablet"]
ad_positions = ["top", "middle", "bottom"]
age_groups = ["18-24", "25-34", "35-44", "45-54", "55+"]
days_of_week = ["poniedzialek", "wtorek", "sroda", "czwartek", "piatek", "sobota", "niedziela"]

ad_category = np.random.choice(ad_categories, N)
time_of_day = np.random.choice(times_of_day, N, p=[0.15, 0.20, 0.25, 0.25, 0.15])
device_type = np.random.choice(device_types, N, p=[0.55, 0.35, 0.10])
ad_position = np.random.choice(ad_positions, N, p=[0.30, 0.40, 0.30])
age_group = np.random.choice(age_groups, N)
day_of_week = np.random.choice(days_of_week, N)

# Realistyczne korelacje: pozycja "top" i urządzenie "mobile" zwiększają CTR
base_prob = 0.05
position_bonus = np.where(ad_position == "top", 0.08, np.where(ad_position == "middle", 0.02, 0.0))
device_bonus = np.where(device_type == "mobile", 0.05, np.where(device_type == "tablet", 0.02, 0.0))
time_bonus = np.where(np.isin(time_of_day, ["popoludnie", "wieczor"]), 0.03, 0.0)
weekend_bonus = np.where(np.isin(day_of_week, ["sobota", "niedziela"]), 0.02, 0.0)

click_prob = np.clip(base_prob + position_bonus + device_bonus + time_bonus + weekend_bonus, 0.0, 0.95)
clicked = (np.random.rand(N) < click_prob).astype(int)

pdf = pd.DataFrame({
    "impression_id": np.arange(1, N + 1),
    "ad_category": ad_category,
    "time_of_day": time_of_day,
    "device_type": device_type,
    "ad_position": ad_position,
    "age_group": age_group,
    "day_of_week": day_of_week,
    "clicked": clicked,
})

df = spark.createDataFrame(pdf)
df.printSchema()
print(f"📊 Liczba wierszy: {df.count():,}")
df.groupBy("clicked").count().show()

## 🎓 Guided Example

Zbudujmy razem pełny `Pipeline` MLlib: `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `LogisticRegression`, a następnie oceńmy go metryką AUC.

**Uwaga:** Ten przykład używa własnego, lokalnego podziału danych (`guided_train` / `guided_test`) tylko do demonstracji. W ćwiczeniach stworzysz **własny** oficjalny podział (`train_df` / `test_df`), którego użyjesz we wszystkich kolejnych zadaniach.

In [ ]:
# @title 🎓 Guided Example — uruchom i przestudiuj

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

categorical_cols = ["ad_category", "time_of_day", "device_type", "ad_position", "age_group", "day_of_week"]
indexed_cols = [f"{c}_idx" for c in categorical_cols]
ohe_cols = [f"{c}_ohe" for c in categorical_cols]

# Krok 1: StringIndexer dla każdej kolumny kategorycznej
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]

# Krok 2: OneHotEncoder dla każdego indeksu (unikamy fałszywej kolejności kategorii)
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]

# Krok 3: VectorAssembler łączy wszystkie zakodowane kolumny w jedną kolumnę "features"
assembler = VectorAssembler(inputCols=ohe_cols, outputCol="features")

# Krok 4: model klasyfikacji
lr = LogisticRegression(featuresCol="features", labelCol="clicked", maxIter=20)

pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

guided_train, guided_test = df.randomSplit([0.7, 0.3], seed=42)

pipeline_model = pipeline.fit(guided_train)          # 👈 trenuje CAŁY łańcuch naraz
guided_predictions = pipeline_model.transform(guided_test)

evaluator = BinaryClassificationEvaluator(
    labelCol="clicked", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
guided_auc = evaluator.evaluate(guided_predictions)

print(f"✅ Guided example — LogisticRegression AUC: {guided_auc:.4f}")
guided_predictions.select("ad_position", "device_type", "clicked", "prediction", "probability").show(5, truncate=False)

---

## ✏️ Ćwiczenie 1: Oficjalny podział train/test

**Trudność:** ⭐

Podziel `df` na `train_df` (70%) i `test_df` (30%), używając ziarna losowości `seed=42`. Te dwie zmienne będą używane we **wszystkich** kolejnych ćwiczeniach.

**Oczekiwany wynik:** `train_df` ma ok. 70% wierszy, `test_df` ok. 30% wierszy, a ich suma równa się `df.count()`.

> 💡 **Hint:** Użyj `df.randomSplit([...], seed=...)`.

In [ ]:
# ✏️ Ćwiczenie 1: Podział train/test

# TODO: podziel df na train_df (70%) i test_df (30%), seed=42
train_df = ...  # Zastąp ... swoim kodem
test_df = ...  # Zastąp ... swoim kodem

print(f"train_df: {train_df.count():,} wierszy")
print(f"test_df: {test_df.count():,} wierszy")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: randomSplit dzieli DataFrame na losowe części wg podanych wag, z ustalonym ziarnem dla powtarzalności
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

print(f"train_df: {train_df.count():,} wierszy")
print(f"test_df: {test_df.count():,} wierszy")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    train_count = train_df.count()
    test_count = test_df.count()
    total = df.count()

    assert train_count is not ..., "❌ train_df nie zostało uzupełnione"
    assert abs(train_count / total - 0.7) < 0.02, f"❌ train_df powinno mieć ok. 70% wierszy, ma {train_count / total:.2%}"
    assert abs(test_count / total - 0.3) < 0.02, f"❌ test_df powinno mieć ok. 30% wierszy, ma {test_count / total:.2%}"
    assert train_count + test_count == total, "❌ suma train_df + test_df powinna równać się df.count()"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 2: Pipeline z RandomForestClassifier

**Trudność:** ⭐⭐

Zbuduj `Pipeline` analogiczny do guided example, ale zamień `LogisticRegression` na `RandomForestClassifier` (`numTrees=100`, `maxDepth=8`). Wytrenuj go na `train_df` i wygeneruj predykcje na `test_df`.

**Oczekiwany wynik:** `rf_predictions` to DataFrame z kolumną `prediction`, o tej samej liczbie wierszy co `test_df`.

> 💡 **Hint:** Możesz ponownie użyć `categorical_cols` i `ohe_cols` z guided example — zmienia się tylko ostatni stage Pipeline'u.

In [ ]:
# ✏️ Ćwiczenie 2: RandomForestClassifier

from pyspark.ml.classification import RandomForestClassifier

rf_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
rf_encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in categorical_cols]
rf_assembler = VectorAssembler(inputCols=ohe_cols, outputCol="features")

# TODO: stwórz RandomForestClassifier(featuresCol="features", labelCol="clicked", numTrees=100, maxDepth=8)
rf_classifier = ...  # Zastąp ... swoim kodem

rf_pipeline = Pipeline(stages=rf_indexers + rf_encoders + [rf_assembler, rf_classifier])

# TODO: wytrenuj pipeline na train_df
rf_model = ...  # Zastąp ... swoim kodem

# TODO: wygeneruj predykcje na test_df
rf_predictions = ...  # Zastąp ... swoim kodem

print(f"Liczba wierszy w rf_predictions: {rf_predictions.count():,}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: ten sam preprocessing (indexers + encoders + assembler), inny model na końcu
rf_classifier = RandomForestClassifier(
    featuresCol="features", labelCol="clicked", numTrees=100, maxDepth=8
)

rf_pipeline = Pipeline(stages=rf_indexers + rf_encoders + [rf_assembler, rf_classifier])

# Krok 2: fit() trenuje CAŁY łańcuch (indexers -> encoders -> assembler -> model) na train_df
rf_model = rf_pipeline.fit(train_df)

# Krok 3: transform() stosuje ten sam wytrenowany łańcuch do test_df
rf_predictions = rf_model.transform(test_df)

print(f"Liczba wierszy w rf_predictions: {rf_predictions.count():,}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    assert rf_classifier is not ..., "❌ rf_classifier nie zostało uzupełnione"
    assert "prediction" in rf_predictions.columns, "❌ Brak kolumny 'prediction' w rf_predictions"
    assert rf_predictions.count() == test_df.count(), "❌ rf_predictions powinno mieć tyle samo wierszy co test_df"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 3: Porównanie AUC — LogisticRegression vs RandomForest

**Trudność:** ⭐⭐

Oblicz AUC dla predykcji `LogisticRegression` (przelicz `pipeline_model` z guided example na oficjalnym `test_df`) oraz dla `rf_predictions` z Ćwiczenia 2. Zapisz wyniki jako `lr_auc` i `rf_auc`.

> 💡 **Hint:** Ponownie użyj `pipeline_model.transform(...)` oraz istniejącego obiektu `evaluator`.

In [ ]:
# ✏️ Ćwiczenie 3: Porównanie AUC

# TODO: wygeneruj predykcje LogisticRegression na oficjalnym test_df
lr_predictions_test = ...  # Zastąp ... swoim kodem

# TODO: oblicz AUC dla obu modeli za pomocą evaluator.evaluate(...)
lr_auc = ...  # Zastąp ... swoim kodem
rf_auc = ...  # Zastąp ... swoim kodem

print(f"LogisticRegression AUC: {lr_auc:.4f}")
print(f"RandomForest AUC:      {rf_auc:.4f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: guided example trenował pipeline_model na guided_train/guided_test — teraz liczymy
# jego predykcje na OFICJALNYM test_df, żeby porównanie z RandomForest było uczciwe (ten sam test set)
lr_predictions_test = pipeline_model.transform(test_df)

# Krok 2: evaluator.evaluate() liczy areaUnderROC z kolumny rawPrediction
lr_auc = evaluator.evaluate(lr_predictions_test)
rf_auc = evaluator.evaluate(rf_predictions)

print(f"LogisticRegression AUC: {lr_auc:.4f}")
print(f"RandomForest AUC:      {rf_auc:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    assert lr_auc is not ..., "❌ lr_auc nie zostało uzupełnione"
    assert rf_auc is not ..., "❌ rf_auc nie zostało uzupełnione"
    assert 0.0 <= lr_auc <= 1.0, "❌ lr_auc powinno być w zakresie [0, 1]"
    assert 0.0 <= rf_auc <= 1.0, "❌ rf_auc powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 4: Ważność cech (feature importances)

**Trudność:** ⭐⭐⭐

Wyciągnij ostatni stage z `rf_model.stages` (obiekt `RandomForestClassificationModel`) i pobierz z niego wektor `featureImportances`. Wypisz liczbę cech oraz 5 największych wartości ważności.

> 💡 **Hint:** `rf_model.stages[-1]` to wytrenowany model lasu losowego. Atrybut `.featureImportances` zwraca `SparseVector`.

In [ ]:
# ✏️ Ćwiczenie 4: Feature importances

# TODO: pobierz ostatni stage pipeline'u (wytrenowany RandomForestClassificationModel)
rf_classification_model = ...  # Zastąp ... swoim kodem

# TODO: pobierz wektor ważności cech
importances = ...  # Zastąp ... swoim kodem

print(f"Liczba cech w modelu: {importances.size}")
print(f"Top-5 wartości ważności: {sorted(importances.toArray(), reverse=True)[:5]}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: rf_model.stages to lista WSZYSTKICH wytrenowanych etapów pipeline'u w kolejności;
# ostatni (-1) to zawsze wytrenowany model (tu: RandomForestClassificationModel)
rf_classification_model = rf_model.stages[-1]

# Krok 2: featureImportances to SparseVector o długości równej liczbie kolumn w "features"
importances = rf_classification_model.featureImportances

print(f"Liczba cech w modelu: {importances.size}")
print(f"Top-5 wartości ważności: {sorted(importances.toArray(), reverse=True)[:5]}")

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    assert importances is not ..., "❌ importances nie zostało uzupełnione"
    assert importances.size > 0, "❌ Wektor ważności cech nie powinien być pusty"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 5: Accuracy, precision i recall

**Trudność:** ⭐⭐⭐

Użyj `MulticlassClassificationEvaluator`, aby obliczyć `accuracy`, `weightedPrecision` i `weightedRecall` dla `rf_predictions` z Ćwiczenia 2.

> 💡 **Hint:** `MulticlassClassificationEvaluator` działa również dla klasyfikacji binarnej — wystarczy ustawić `labelCol` i `predictionCol`.

In [ ]:
# ✏️ Ćwiczenie 5: Accuracy, precision, recall

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="accuracy"
)
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="weightedPrecision"
)
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="clicked", predictionCol="prediction", metricName="weightedRecall"
)

# TODO: oblicz metryki na rf_predictions używając powyższych evaluatorów
accuracy = ...  # Zastąp ... swoim kodem
precision = ...  # Zastąp ... swoim kodem
recall = ...  # Zastąp ... swoim kodem

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: każdy evaluator.evaluate(predictions) liczy jedną metrykę na podstawie kolumn
# "clicked" (prawdziwa etykieta) i "prediction" (przewidziana klasa 0/1)
accuracy = accuracy_evaluator.evaluate(rf_predictions)
precision = precision_evaluator.evaluate(rf_predictions)
recall = recall_evaluator.evaluate(rf_predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    for name, value in [("accuracy", accuracy), ("precision", precision), ("recall", recall)]:
        assert value is not ..., f"❌ {name} nie zostało uzupełnione"
        assert 0.0 <= value <= 1.0, f"❌ {name} powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 6: Inspekcja DataFrame z predykcjami

**Trudność:** ⭐⭐⭐

Wybierz z `rf_predictions` kolumny `ad_position`, `device_type`, `clicked`, `prediction` i `probability`, i wyświetl 10 przykładowych wierszy jako `sample_predictions`.

> 💡 **Hint:** Użyj `.select(...)` i `.show(10, truncate=False)`.

In [ ]:
# ✏️ Ćwiczenie 6: Inspekcja predykcji

# TODO: wybierz kolumny ad_position, device_type, clicked, prediction, probability z rf_predictions
sample_predictions = ...  # Zastąp ... swoim kodem

sample_predictions.show(10, truncate=False)

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: .select() wybiera tylko interesujące kolumny, żeby łatwiej przeglądać wyniki
sample_predictions = rf_predictions.select(
    "ad_position", "device_type", "clicked", "prediction", "probability"
)

sample_predictions.show(10, truncate=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    expected_cols = {"ad_position", "device_type", "clicked", "prediction"}
    assert expected_cols.issubset(set(sample_predictions.columns)), "❌ Brakuje wymaganych kolumn w sample_predictions"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 7: Wpływ pozycji reklamy i urządzenia na CTR

**Trudność:** ⭐⭐⭐⭐

Pogrupuj `rf_predictions` po `ad_position` i `device_type`, i policz **średnią rzeczywistą** wartość `clicked` (`rzeczywisty_ctr`) oraz **średnią przewidywaną** wartość `prediction` (`przewidywany_ctr`). Posortuj malejąco po `rzeczywisty_ctr` jako `position_effect`.

**Oczekiwany wynik:** kombinacja `ad_position="top"` + `device_type="mobile"` powinna mieć wyższy `rzeczywisty_ctr` niż `ad_position="bottom"` + `device_type="desktop"`.

> 💡 **Hint:** Użyj `F.avg("clicked")` i `F.avg("prediction")` w `.agg(...)`.

In [ ]:
# ✏️ Ćwiczenie 7: Wpływ pozycji reklamy na CTR

from pyspark.sql import functions as F

# TODO: pogrupuj rf_predictions po ad_position i device_type, policz średni clicked i średnią prediction
position_effect = ...  # Zastąp ... swoim kodem

position_effect.show()

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 7 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: groupBy + agg liczy średnią rzeczywistą i przewidywaną wartość CTR per kombinacja
position_effect = (
    rf_predictions
    .groupBy("ad_position", "device_type")
    .agg(
        F.avg("clicked").alias("rzeczywisty_ctr"),
        F.avg("prediction").alias("przewidywany_ctr"),
    )
    # Krok 2: sortujemy malejąco, żeby najlepsze kombinacje (najwyższy realny CTR) były na górze
    .orderBy(F.col("rzeczywisty_ctr").desc())
)

position_effect.show()

In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    expected_cols = {"ad_position", "device_type", "rzeczywisty_ctr", "przewidywany_ctr"}
    assert expected_cols.issubset(set(position_effect.columns)), "❌ Brakuje wymaganych kolumn w position_effect"

    rows = {(r["ad_position"], r["device_type"]): r["rzeczywisty_ctr"] for r in position_effect.collect()}
    top_mobile = rows.get(("top", "mobile"))
    bottom_desktop = rows.get(("bottom", "desktop"))

    assert top_mobile is not None and bottom_desktop is not None, "❌ Brakuje oczekiwanych kombinacji w wyniku"
    assert top_mobile > bottom_desktop, "❌ 'top' + 'mobile' powinno mieć wyższy CTR niż 'bottom' + 'desktop'"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## 🏆 Bonus Challenge (opcjonalnie)

Zapisz wytrenowany `rf_model` (`PipelineModel`) na dysk, wczytaj go ponownie i zweryfikuj, że predykcje po wczytaniu są identyczne co do liczby wierszy z oryginalnymi `rf_predictions`.

> 💡 **Hint:** `PipelineModel` ma metody `.write().overwrite().save(path)` oraz klasową metodę `PipelineModel.load(path)`.

In [ ]:
# 🏆 Bonus: zapis i odczyt modelu

from pyspark.ml import PipelineModel

model_path = "/tmp/ctr_rf_pipeline_model"

# TODO: zapisz rf_model pod model_path (nadpisując, jeśli już istnieje)
...

# TODO: wczytaj model z powrotem jako loaded_model
loaded_model = ...  # Zastąp ... swoim kodem

reloaded_predictions = loaded_model.transform(test_df)

assert reloaded_predictions.count() == rf_predictions.count(), "❌ Liczba wierszy po wczytaniu modelu się nie zgadza"
print("✅ Model poprawnie zapisany i wczytany! 🎉")

In [ ]:
# @title 💡 Pełne rozwiązanie — Bonus (kliknij, aby zobaczyć krok po kroku)

from pyspark.ml import PipelineModel

model_path = "/tmp/ctr_rf_pipeline_model"

# Krok 1: write().overwrite().save() zapisuje CAŁY PipelineModel (wszystkie etapy) na dysk/HDFS/GCS
rf_model.write().overwrite().save(model_path)

# Krok 2: PipelineModel.load() odtwarza dokładnie ten sam, już wytrenowany łańcuch
loaded_model = PipelineModel.load(model_path)

reloaded_predictions = loaded_model.transform(test_df)

assert reloaded_predictions.count() == rf_predictions.count(), "❌ Liczba wierszy po wczytaniu modelu się nie zgadza"
print("✅ Model poprawnie zapisany i wczytany! 🎉")

---

## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Kodowanie zmiennych kategorycznych `StringIndexer` + `OneHotEncoder`
- ✅ Budowę pełnego `Pipeline` MLlib (preprocessing + model)
- ✅ Trening i porównanie `LogisticRegression` oraz `RandomForestClassifier`
- ✅ Ewaluację modeli: AUC, accuracy, precision, recall
- ✅ Analizę ważności cech i predykcji modelu
- ✅ Zapis i odczyt wytrenowanego `PipelineModel`

### 📚 Dalsze materiały
- [Spark MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html)
- [MLlib API Reference (pyspark.ml)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html)

### 🔮 Co dalej?
W **Części B** (poniżej, ten sam notebook) poznasz zaawansowany MLlib — **tuning hiperparametrów** za pomocą `ParamGridBuilder` i `CrossValidator`!

---

# 🅱️ Część B: MLlib — Wykrywanie fraudów i tuning hiperparametrów 💳🕵️

Pracujesz jako data scientist w firmie **CardShield**, dostawcy systemów antyfraudowych dla banków. Masz ~500 000 transakcji, z których tylko 1-2% to fraudy — klasyczny przypadek **class imbalance** w skali Big Data. Ta część generuje **własny, niezależny** zbiór danych (nie korzysta z danych Części A).


In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

import random
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession

random.seed(42)
np.random.seed(42)

spark = (
    SparkSession.builder
    .appName("CardShield-Fraud-Detection")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("✅ Setup complete! SparkSession gotowa 🚀")
print(f"Wersja Sparka: {spark.version}")

## 📖 Kontekst: CardShield 💳🕵️

Pracujesz jako data scientist w firmie **CardShield**, dostawcy systemów antyfraudowych dla banków. Masz ~500 000 transakcji kartą płatniczą z ostatniego miesiąca — ale tylko **1-2%** z nich to fraudy. To klasyczny problem **class imbalance** w skali Big Data: model, który zawsze przewiduje "nie fraud", osiągnie >98% accuracy i będzie kompletnie bezużyteczny!

### Kluczowe koncepty na dziś

| Koncept | Opis | Przykład |
|---------|------|----------|
| `ParamGridBuilder` | buduje siatkę kombinacji hiperparametrów do przetestowania | `ParamGridBuilder().addGrid(rf.numTrees, [50, 100]).build()` |
| `CrossValidator` | trenuje i ocenia model dla każdej kombinacji z siatki, k-fold CV | `CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3)` |
| `TrainValidationSplit` | jak `CrossValidator`, ale pojedynczy podział zamiast k-fold — szybszy, mniej solidny | `TrainValidationSplit(..., trainRatio=0.8)` |
| `weightCol` | kolumna wag przykładów — sposób na class imbalance bez fizycznego undersamplingu | `RandomForestClassifier(weightCol="class_weight")` |
| `areaUnderPR` | AUC dla krzywej Precision-Recall — lepsza metryka niż AUC-ROC przy silnym imbalance | `BinaryClassificationEvaluator(metricName="areaUnderPR")` |

> 🔑 **Zasada kciuka:** Gdy klasa pozytywna to <5% danych, **AUC-PR** mówi Ci znacznie więcej niż AUC-ROC — bo skupia się na tym, jak dobrze model radzi sobie właśnie z rzadką klasą.

In [ ]:
# @title 📊 Generowanie zbioru danych transakcji (~500 000 wierszy, fraud rate ~1-2%)

N = 500_000

merchant_categories = ["elektronika", "paliwo", "gastronomia", "podroze", "apteka", "odziez", "elektronika_online", "kasyno_online"]
countries = ["PL", "DE", "FR", "US", "GB", "NG", "RU", "BR"]

amount = np.round(np.random.lognormal(mean=3.8, sigma=1.1, size=N), 2)
hour = np.random.randint(0, 24, N)
card_country = np.random.choice(countries, N, p=[0.55, 0.15, 0.08, 0.08, 0.06, 0.03, 0.03, 0.02])

# transaction_country zwykle = card_country, ale czasem różny kraj (sygnał fraudu)
different_country_mask = np.random.rand(N) < 0.08
transaction_country = np.where(
    different_country_mask,
    np.random.choice(countries, N),
    card_country,
)

merchant_category = np.random.choice(merchant_categories, N, p=[0.18, 0.15, 0.20, 0.10, 0.12, 0.13, 0.07, 0.05])
transactions_last_hour = np.random.poisson(lam=1.2, size=N)

# Realistyczne sygnały fraudu: inny kraj, wysoka kwota, kasyno_online, dużo transakcji w ostatniej godzinie, nocna pora
base_prob = 0.006
country_mismatch_bonus = np.where(transaction_country != card_country, 0.06, 0.0)
high_amount_bonus = np.where(amount > 300, 0.03, 0.0)
casino_bonus = np.where(merchant_category == "kasyno_online", 0.05, 0.0)
velocity_bonus = np.where(transactions_last_hour >= 4, 0.08, 0.0)
night_bonus = np.where((hour >= 1) & (hour <= 4), 0.02, 0.0)

fraud_prob = np.clip(
    base_prob + country_mismatch_bonus + high_amount_bonus + casino_bonus + velocity_bonus + night_bonus,
    0.0, 0.97,
)
is_fraud = (np.random.rand(N) < fraud_prob).astype(int)

pdf = pd.DataFrame({
    "transaction_id": np.arange(1, N + 1),
    "amount": amount,
    "hour": hour,
    "transaction_country": transaction_country,
    "card_country": card_country,
    "merchant_category": merchant_category,
    "transactions_last_hour": transactions_last_hour,
    "is_fraud": is_fraud,
})

df = spark.createDataFrame(pdf)
df.printSchema()
print(f"📊 Liczba wierszy: {df.count():,}")
df.groupBy("is_fraud").count().show()
fraud_rate = df.filter("is_fraud = 1").count() / df.count()
print(f"🕵️ Fraud rate: {fraud_rate:.2%}")

## 🎓 Guided Example — Pipeline + CrossValidator z ParamGridBuilder

Zbudujmy razem pełny `Pipeline` MLlib (`StringIndexer` → `VectorAssembler` → `RandomForestClassifier`), a następnie owińmy go w `CrossValidator` z małą siatką hiperparametrów, oceniając modele metryką **areaUnderPR** (bo dane są mocno niezbalansowane!).

**Uwaga:** Ten przykład używa własnego, lokalnego podziału danych (`guided_train` / `guided_test`) tylko do demonstracji. W ćwiczeniach stworzysz **własny** oficjalny podział (`train_df` / `test_df`), którego użyjesz we wszystkich kolejnych zadaniach.

In [ ]:
# @title 🎓 Guided Example — uruchom i przestudiuj (może potrwać kilka minut na CrossValidator)

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

categorical_cols = ["transaction_country", "card_country", "merchant_category"]
indexed_cols = [f"{c}_idx" for c in categorical_cols]
numeric_cols = ["amount", "hour", "transactions_last_hour"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="is_fraud", seed=42)

guided_pipeline = Pipeline(stages=indexers + [assembler, rf])

# Mała siatka — na zajęciach chcemy szybki wynik, nie produkcyjny grid search!
guided_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)

guided_cv = CrossValidator(
    estimator=guided_pipeline,
    estimatorParamMaps=guided_grid,
    evaluator=pr_evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

guided_train, guided_test = df.randomSplit([0.7, 0.3], seed=42)
guided_train = guided_train.cache()          # 👈 CrossValidator wielokrotnie odczytuje train — cache przyspiesza CV

guided_cv_model = guided_cv.fit(guided_train)
guided_predictions = guided_cv_model.bestModel.transform(guided_test)

guided_auc_pr = pr_evaluator.evaluate(guided_predictions)
print(f"✅ Guided example — najlepszy model, AUC-PR: {guided_auc_pr:.4f}")
guided_predictions.select("amount", "transaction_country", "card_country", "is_fraud", "prediction", "probability").show(5, truncate=False)

---

## ✏️ Ćwiczenie 1: Oficjalny podział train/test

**Trudność:** ⭐

Podziel `df` na `train_df` (70%) i `test_df` (30%), używając ziarna losowości `seed=42`. Zcache'uj `train_df` — będzie wielokrotnie wykorzystywany przez `CrossValidator` w kolejnych ćwiczeniach. Te dwie zmienne będą używane we **wszystkich** kolejnych ćwiczeniach.

**Oczekiwany wynik:** `train_df` ma ok. 70% wierszy, `test_df` ok. 30% wierszy, a ich suma równa się `df.count()`.

> 💡 **Hint:** Użyj `df.randomSplit([...], seed=...)`, a potem `.cache()` na `train_df`.

In [ ]:
# ✏️ Ćwiczenie 1: Podział train/test

# TODO: podziel df na train_df (70%) i test_df (30%), seed=42, zcache'uj train_df
train_df = ...  # Zastąp ... swoim kodem
test_df = ...  # Zastąp ... swoim kodem

print(f"train_df: {train_df.count():,} wierszy")
print(f"test_df: {test_df.count():,} wierszy")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: ten sam podział 70/30 co w Części A, ale na NOWYM df (transakcje, nie reklamy)
train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

# Krok 2: cache() trzyma train_df w pamięci po pierwszym odczycie — CrossValidator będzie
# czytał train_df wielokrotnie (raz na fold x kombinację hiperparametrów), więc to spore przyspieszenie
train_df = train_df.cache()

print(f"train_df: {train_df.count():,} wierszy")
print(f"test_df: {test_df.count():,} wierszy")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    train_count = train_df.count()
    test_count = test_df.count()
    total = df.count()

    assert train_count is not ..., "❌ train_df nie zostało uzupełnione"
    assert abs(train_count / total - 0.7) < 0.02, f"❌ train_df powinno mieć ok. 70% wierszy, ma {train_count / total:.2%}"
    assert abs(test_count / total - 0.3) < 0.02, f"❌ test_df powinno mieć ok. 30% wierszy, ma {test_count / total:.2%}"
    assert train_count + test_count == total, "❌ suma train_df + test_df powinna równać się df.count()"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 2: AUC-ROC vs AUC-PR — dlaczego to ma znaczenie?

**Trudność:** ⭐⭐

Weź `guided_predictions` z guided example. Oblicz **obie** metryki: `areaUnderROC` i `areaUnderPR` przy pomocy `BinaryClassificationEvaluator`. Zapisz je jako `auc_roc` i `auc_pr`. Następnie w zmiennej `wyjasnienie` (string, min. 40 znaków) napisz po polsku, dlaczego przy fraud rate ~1-2% AUC-PR jest bardziej wiarygodną metryką niż AUC-ROC.

**Oczekiwany wynik:** `auc_roc` będzie zauważalnie wyższe niż `auc_pr` — to normalne przy silnym class imbalance!

> 💡 **Hint:** AUC-ROC uwzględnia True Negative Rate, którego przy 98-99% klasy negatywnej łatwo "osiągnąć przypadkiem". AUC-PR skupia się tylko na precision/recall klasy pozytywnej.

In [ ]:
# ✏️ Ćwiczenie 2: Porównanie AUC-ROC vs AUC-PR

# TODO: stwórz evaluator z metricName="areaUnderROC" i policz auc_roc na guided_predictions
auc_roc = ...  # Zastąp ... swoim kodem

# TODO: policz auc_pr (może użyć już istniejącego pr_evaluator z guided example)
auc_pr = ...  # Zastąp ... swoim kodem

# TODO: krótkie wyjaśnienie po polsku (min. 40 znaków)
wyjasnienie = "..."  # Zastąp ... swoim tekstem

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")
print(f"Wyjaśnienie: {wyjasnienie}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: osobny evaluator z metricName="areaUnderROC" na tych samych predykcjach
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
auc_roc = roc_evaluator.evaluate(guided_predictions)

# Krok 2: pr_evaluator już istnieje z guided example (metricName="areaUnderPR")
auc_pr = pr_evaluator.evaluate(guided_predictions)

# Krok 3: wyjaśnienie — przy 1-2% fraudów AUC-ROC jest "zawyżane" przez ogromną liczbę True Negatives
wyjasnienie = (
    "Przy fraud rate ~1-2% AUC-ROC jest zawyżane, bo model osiąga wysoki TNR niemal za darmo "
    "(dominująca klasa negatywna). AUC-PR skupia się tylko na precision/recall rzadkiej klasy pozytywnej, "
    "więc lepiej odzwierciedla realną jakość wykrywania fraudów."
)

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")
print(f"Wyjaśnienie: {wyjasnienie}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    assert auc_roc is not ..., "❌ auc_roc nie zostało uzupełnione"
    assert auc_pr is not ..., "❌ auc_pr nie zostało uzupełnione"
    assert 0.0 <= auc_roc <= 1.0, "❌ auc_roc powinno być w zakresie [0, 1]"
    assert 0.0 <= auc_pr <= 1.0, "❌ auc_pr powinno być w zakresie [0, 1]"
    assert wyjasnienie != "..." and len(wyjasnienie) >= 40, "❌ wyjasnienie powinno mieć min. 40 znaków"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 3: Ważenie klas (`weightCol`)

**Trudność:** ⭐⭐⭐

Dodaj do `train_df` kolumnę `class_weight`: `1.0` dla `is_fraud == 0`, oraz `(liczba nie-fraudów / liczba fraudów)` dla `is_fraud == 1` (klasyczne *inverse class frequency weighting*). Zapisz wynik jako `train_weighted`. Następnie zbuduj `Pipeline` analogiczny do guided example, ale z `RandomForestClassifier(weightCol="class_weight", ...)`, wytrenuj go i policz `auc_pr_weighted` na `test_df`.

**Oczekiwany wynik:** `train_weighted` ma dodatkową kolumnę `class_weight`; `auc_pr_weighted` to liczba w [0, 1].

> 💡 **Hint:** Użyj `F.when(F.col("is_fraud") == 1, waga_fraud).otherwise(1.0)`.

In [ ]:
# ✏️ Ćwiczenie 3: Ważenie klas

from pyspark.sql import functions as F

n_fraud = train_df.filter("is_fraud = 1").count()
n_ok = train_df.filter("is_fraud = 0").count()
fraud_weight = n_ok / n_fraud

# TODO: dodaj kolumnę class_weight do train_df -> train_weighted
train_weighted = ...  # Zastąp ... swoim kodem

weighted_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
weighted_assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")

# TODO: stwórz RandomForestClassifier z weightCol="class_weight", numTrees=100, maxDepth=8
weighted_rf = ...  # Zastąp ... swoim kodem

weighted_pipeline = Pipeline(stages=weighted_indexers + [weighted_assembler, weighted_rf])

# TODO: wytrenuj na train_weighted i policz auc_pr_weighted na test_df
weighted_model = ...  # Zastąp ... swoim kodem
auc_pr_weighted = ...  # Zastąp ... swoim kodem

print(f"Waga klasy fraud: {fraud_weight:.1f}")
print(f"AUC-PR (ważony model): {auc_pr_weighted:.4f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

from pyspark.sql import functions as F

n_fraud = train_df.filter("is_fraud = 1").count()
n_ok = train_df.filter("is_fraud = 0").count()
fraud_weight = n_ok / n_fraud

# Krok 1: nie-fraudy mają wagę 1.0, fraudy dostają wagę (liczba_ok / liczba_fraud) —
# im rzadsza klasa, tym większa "kara" za jej pomylenie podczas treningu
train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, fraud_weight).otherwise(1.0),
)

weighted_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
weighted_assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")

# Krok 2: weightCol mówi modelowi, żeby "ważył" błędy na każdym wierszu tą kolumną podczas treningu
weighted_rf = RandomForestClassifier(
    featuresCol="features", labelCol="is_fraud", weightCol="class_weight",
    numTrees=100, maxDepth=8, seed=42,
)

weighted_pipeline = Pipeline(stages=weighted_indexers + [weighted_assembler, weighted_rf])

# Krok 3: trenujemy na train_weighted (z kolumną wag), oceniamy na niezmienionym test_df
weighted_model = weighted_pipeline.fit(train_weighted)
auc_pr_weighted = pr_evaluator.evaluate(weighted_model.transform(test_df))

print(f"Waga klasy fraud: {fraud_weight:.1f}")
print(f"AUC-PR (ważony model): {auc_pr_weighted:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    assert "class_weight" in train_weighted.columns, "❌ train_weighted powinno mieć kolumnę class_weight"
    assert auc_pr_weighted is not ..., "❌ auc_pr_weighted nie zostało uzupełnione"
    assert 0.0 <= auc_pr_weighted <= 1.0, "❌ auc_pr_weighted powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 4: `GBTClassifier` z `CrossValidator`

**Trudność:** ⭐⭐⭐⭐

Zbuduj `Pipeline` z `GBTClassifier` (zamiast `RandomForestClassifier`), owiń go w `CrossValidator` z małą siatką `ParamGridBuilder` (np. `maxIter` w `[20, 50]`, `maxDepth` w `[3, 5]`), `numFolds=3`, oceniając `areaUnderPR`. Wytrenuj na `train_df` i zapisz najlepszy model jako `gbt_cv_model`, a AUC-PR na `test_df` jako `auc_pr_gbt`.

**Oczekiwany wynik:** `gbt_cv_model.bestModel` to `PipelineModel`; `auc_pr_gbt` to liczba w [0, 1].

> ⚠️ **Uwaga:** `GBTClassifier` z `CrossValidator` na 500k wierszy może potrwać kilka minut — to normalne, taki jest koszt tuningu na dużych danych!

> 💡 **Hint:** `GBTClassifier` obsługuje tylko klasyfikację binarną — idealnie pasuje do `is_fraud`.

In [ ]:
# ✏️ Ćwiczenie 4: GBTClassifier + CrossValidator

from pyspark.ml.classification import GBTClassifier

gbt_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
gbt_assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")

# TODO: stwórz GBTClassifier(featuresCol="features", labelCol="is_fraud", seed=42)
gbt = ...  # Zastąp ... swoim kodem

gbt_pipeline = Pipeline(stages=gbt_indexers + [gbt_assembler, gbt])

# TODO: zbuduj małą siatkę ParamGridBuilder: gbt.maxIter w [20, 50], gbt.maxDepth w [3, 5]
gbt_grid = ...  # Zastąp ... swoim kodem

# TODO: stwórz CrossValidator (estimator=gbt_pipeline, estimatorParamMaps=gbt_grid, evaluator=pr_evaluator, numFolds=3, parallelism=2, seed=42)
gbt_cv = ...  # Zastąp ... swoim kodem

# TODO: wytrenuj na train_df -> gbt_cv_model
gbt_cv_model = ...  # Zastąp ... swoim kodem

gbt_predictions = gbt_cv_model.bestModel.transform(test_df)
auc_pr_gbt = pr_evaluator.evaluate(gbt_predictions)
print(f"AUC-PR (GBTClassifier, najlepszy z CV): {auc_pr_gbt:.4f}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

from pyspark.ml.classification import GBTClassifier

gbt_indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
gbt_assembler = VectorAssembler(inputCols=indexed_cols + numeric_cols, outputCol="features")

# Krok 1: GBTClassifier wspiera tylko klasyfikację binarną — idealny dla is_fraud (0/1)
gbt = GBTClassifier(featuresCol="features", labelCol="is_fraud", seed=42)

gbt_pipeline = Pipeline(stages=gbt_indexers + [gbt_assembler, gbt])

# Krok 2: mała siatka 2x2=4 kombinacji — każda zostanie oceniona 3-krotnie (numFolds=3), czyli 12 treningów
gbt_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [20, 50])
    .addGrid(gbt.maxDepth, [3, 5])
    .build()
)

# Krok 3: CrossValidator trenuje gbt_pipeline dla każdej kombinacji z gbt_grid, z 3-fold CV,
# i ocenia areaUnderPR (dobrana wcześniej metryka dla niezbalansowanych danych)
gbt_cv = CrossValidator(
    estimator=gbt_pipeline,
    estimatorParamMaps=gbt_grid,
    evaluator=pr_evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

# Krok 4: fit() trenuje wszystkie kombinacje i automatycznie wybiera najlepszą (bestModel)
gbt_cv_model = gbt_cv.fit(train_df)

gbt_predictions = gbt_cv_model.bestModel.transform(test_df)
auc_pr_gbt = pr_evaluator.evaluate(gbt_predictions)
print(f"AUC-PR (GBTClassifier, najlepszy z CV): {auc_pr_gbt:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    from pyspark.ml import PipelineModel
    assert isinstance(gbt_cv_model.bestModel, PipelineModel), "❌ gbt_cv_model.bestModel powinno być PipelineModel"
    assert 0.0 <= auc_pr_gbt <= 1.0, "❌ auc_pr_gbt powinno być w zakresie [0, 1]"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 5: Ekstrakcja najlepszych hiperparametrów z CV

**Trudność:** ⭐⭐⭐

Z `gbt_cv_model` wyciągnij ostatni stage najlepszego modelu (`GBTClassificationModel`) i odczytaj z niego wybrane przez `CrossValidator` wartości `maxIter` i `maxDepth`. Zapisz je jako `best_max_iter` i `best_max_depth`.

**Oczekiwany wynik:** obie wartości pochodzą z siatki, którą zdefiniowałeś/aś w Ćwiczeniu 4.

> 💡 **Hint:** `gbt_cv_model.bestModel.stages[-1]` to wytrenowany model; użyj `.getMaxIter()` i `.getMaxDepth()`.

In [ ]:
# ✏️ Ćwiczenie 5: Najlepsze hiperparametry z CV

# TODO: wyciągnij ostatni stage najlepszego modelu
best_gbt_model = ...  # Zastąp ... swoim kodem

# TODO: odczytaj maxIter i maxDepth z best_gbt_model
best_max_iter = ...  # Zastąp ... swoim kodem
best_max_depth = ...  # Zastąp ... swoim kodem

print(f"Najlepszy maxIter: {best_max_iter}")
print(f"Najlepszy maxDepth: {best_max_depth}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: bestModel to PipelineModel — ostatni stage to wytrenowany GBTClassificationModel
best_gbt_model = gbt_cv_model.bestModel.stages[-1]

# Krok 2: gettery .getMaxIter()/.getMaxDepth() odczytują wartości PARAMETRÓW, z którymi
# ten konkretny model został wytrenowany (czyli te, które CrossValidator uznał za najlepsze)
best_max_iter = best_gbt_model.getMaxIter()
best_max_depth = best_gbt_model.getMaxDepth()

print(f"Najlepszy maxIter: {best_max_iter}")
print(f"Najlepszy maxDepth: {best_max_depth}")

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    assert best_max_iter in [20, 50], f"❌ best_max_iter powinno pochodzić z siatki [20, 50], jest {best_max_iter}"
    assert best_max_depth in [3, 5], f"❌ best_max_depth powinno pochodzić z siatki [3, 5], jest {best_max_depth}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 6: Zapis i odczyt najlepszego `PipelineModel`

**Trudność:** ⭐⭐

Zapisz `gbt_cv_model.bestModel` do ścieżki `"/tmp/cardshield_best_model"` (użyj `.write().overwrite().save(...)`), a następnie wczytaj go z powrotem jako `loaded_model` przy pomocy `PipelineModel.load(...)`. Zweryfikuj, że predykcje wczytanego modelu na `test_df` są identyczne jak przed zapisem.

**Oczekiwany wynik:** `loaded_model` generuje te same predykcje co `gbt_cv_model.bestModel`.

> 💡 **Hint:** `PipelineModel` (i wszystkie stage'y) implementuje `MLWritable`/`MLReadable`.

In [ ]:
# ✏️ Ćwiczenie 6: Zapis i odczyt modelu

model_path = "/tmp/cardshield_best_model"

# TODO: zapisz gbt_cv_model.bestModel pod model_path (nadpisz jeśli istnieje)
...  # Zastąp ... swoim kodem

# TODO: wczytaj model z powrotem jako loaded_model
loaded_model = ...  # Zastąp ... swoim kodem

original_preds = gbt_cv_model.bestModel.transform(test_df).select("transaction_id", "prediction")
loaded_preds = loaded_model.transform(test_df).select("transaction_id", "prediction")

mismatches = original_preds.join(loaded_preds, "transaction_id", "inner") \
    .filter(original_preds["prediction"] != loaded_preds["prediction"]).count()
print(f"Liczba niezgodnych predykcji: {mismatches}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

from pyspark.ml import PipelineModel

model_path = "/tmp/cardshield_best_model"

# Krok 1: zapisujemy CAŁY najlepszy PipelineModel (indexers + assembler + GBT) na dysk
gbt_cv_model.bestModel.write().overwrite().save(model_path)

# Krok 2: wczytujemy go z powrotem jako nowy, niezależny obiekt
loaded_model = PipelineModel.load(model_path)

original_preds = gbt_cv_model.bestModel.transform(test_df).select("transaction_id", "prediction")
loaded_preds = loaded_model.transform(test_df).select("transaction_id", "prediction")

mismatches = original_preds.join(loaded_preds, "transaction_id", "inner") \
    .filter(original_preds["prediction"] != loaded_preds["prediction"]).count()
print(f"Liczba niezgodnych predykcji: {mismatches}")

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    from pyspark.ml import PipelineModel
    assert isinstance(loaded_model, PipelineModel), "❌ loaded_model powinno być PipelineModel"
    assert mismatches == 0, f"❌ Wczytany model powinien dawać identyczne predykcje, znaleziono {mismatches} różnic"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## ✏️ Ćwiczenie 7: Macierz konfuzji przy niestandardowym progu

**Trudność:** ⭐⭐⭐⭐

Domyślny próg klasyfikacji w Spark to 0.5 — ale przy fraud rate ~1-2% to zwykle zbyt wysoko! Wyciągnij prawdopodobieństwo klasy `1` z kolumny `probability` (to `Vector`, drugi element) do nowej kolumny `prob_fraud`, a następnie stwórz kolumnę `custom_prediction` = `1`, jeśli `prob_fraud > 0.3`, w przeciwnym razie `0`. Policz macierz konfuzji (TP, FP, TN, FN) jako słownik `confusion` na `test_df` przetransformowanym przez `gbt_cv_model.bestModel`.

**Oczekiwany wynik:** `confusion` to słownik z kluczami `"TP"`, `"FP"`, `"TN"`, `"FN"`, których suma = liczba wierszy `test_df`.

> 💡 **Hint:** Użyj UDF-a lub `vector_to_array` z `pyspark.ml.functions`, by wyciągnąć element wektora `probability`.

In [ ]:
# ✏️ Ćwiczenie 7: Macierz konfuzji przy progu 0.3

from pyspark.ml.functions import vector_to_array

threshold_predictions = gbt_cv_model.bestModel.transform(test_df)

# TODO: dodaj kolumnę prob_fraud = drugi element wektora "probability" (indeks 1)
threshold_predictions = threshold_predictions.withColumn(
    "prob_fraud", ...  # Zastąp ... swoim kodem
)

# TODO: dodaj kolumnę custom_prediction = 1, gdy prob_fraud > 0.3, inaczej 0
threshold_predictions = threshold_predictions.withColumn(
    "custom_prediction", ...  # Zastąp ... swoim kodem
)

tp = threshold_predictions.filter("is_fraud = 1 AND custom_prediction = 1").count()
fp = threshold_predictions.filter("is_fraud = 0 AND custom_prediction = 1").count()
tn = threshold_predictions.filter("is_fraud = 0 AND custom_prediction = 0").count()
fn = threshold_predictions.filter("is_fraud = 1 AND custom_prediction = 0").count()

confusion = {"TP": tp, "FP": fp, "TN": tn, "FN": fn}
print(f"Macierz konfuzji (próg 0.3): {confusion}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 7 (kliknij, aby zobaczyć krok po kroku)

from pyspark.ml.functions import vector_to_array

threshold_predictions = gbt_cv_model.bestModel.transform(test_df)

# Krok 1: probability to wektor [P(klasa=0), P(klasa=1)] — bierzemy indeks 1 (prawdopodobieństwo fraudu)
threshold_predictions = threshold_predictions.withColumn(
    "prob_fraud", vector_to_array("probability")[1]
)

# Krok 2: niższy próg (0.3 zamiast domyślnych 0.5) oznacza, że łapiemy więcej podejrzanych transakcji
threshold_predictions = threshold_predictions.withColumn(
    "custom_prediction", (F.col("prob_fraud") > 0.3).cast("int")
)

tp = threshold_predictions.filter("is_fraud = 1 AND custom_prediction = 1").count()
fp = threshold_predictions.filter("is_fraud = 0 AND custom_prediction = 1").count()
tn = threshold_predictions.filter("is_fraud = 0 AND custom_prediction = 0").count()
fn = threshold_predictions.filter("is_fraud = 1 AND custom_prediction = 0").count()

confusion = {"TP": tp, "FP": fp, "TN": tn, "FN": fn}
print(f"Macierz konfuzji (próg 0.3): {confusion}")

In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    assert set(confusion.keys()) == {"TP", "FP", "TN", "FN"}, "❌ confusion powinno mieć klucze TP, FP, TN, FN"
    assert sum(confusion.values()) == test_df.count(), "❌ suma macierzy konfuzji powinna równać się liczbie wierszy test_df"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---

## 🏆 Bonus Challenge (opcjonalne)

Bank szacuje, że koszt **niewykrytego fraudu (FN)** to średnio **500 zł** (zwrot pieniędzy klientowi), a koszt **fałszywego alarmu (FP)** to **20 zł** (koszt obsługi + niezadowolenie klienta, którego transakcję wstrzymano). Używając `confusion` z Ćwiczenia 7, policz całkowity koszt (`total_cost = FN * 500 + FP * 20`) dla progu 0.3, a następnie powtórz obliczenia macierzy konfuzji i kosztu dla progu **0.1** i **0.5**. Który próg minimalizuje koszt? Zapisz wniosek w komórce Markdown.

In [ ]:
# 🏆 Bonus: analiza kosztowo-korzyściowa progu decyzyjnego

def confusion_at_threshold(predictions_df, threshold: float) -> dict[str, int]:
    preds = predictions_df.withColumn("prob_fraud", vector_to_array("probability")[1])
    preds = preds.withColumn("custom_prediction", (F.col("prob_fraud") > threshold).cast("int"))
    tp = preds.filter("is_fraud = 1 AND custom_prediction = 1").count()
    fp = preds.filter("is_fraud = 0 AND custom_prediction = 1").count()
    tn = preds.filter("is_fraud = 0 AND custom_prediction = 0").count()
    fn = preds.filter("is_fraud = 1 AND custom_prediction = 0").count()
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn}

base_predictions = gbt_cv_model.bestModel.transform(test_df)

for t in [0.1, 0.3, 0.5]:
    c = confusion_at_threshold(base_predictions, t)
    cost = c["FN"] * 500 + c["FP"] * 20
    print(f"Próg {t}: {c} -> koszt = {cost:,} zł")

---

## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Tuning hiperparametrów przy pomocy `ParamGridBuilder` i `CrossValidator`
- ✅ Wybór odpowiedniej metryki (AUC-PR zamiast AUC-ROC) dla mocno niezbalansowanych danych
- ✅ Ważenie klas (`weightCol`) jako alternatywę dla undersamplingu
- ✅ Trenowanie i ewaluację `GBTClassifier` na dużym zbiorze danych
- ✅ Ekstrakcję najlepszych hiperparametrów z modelu wybranego przez CV
- ✅ Zapis i odczyt `PipelineModel`
- ✅ Analizę macierzy konfuzji przy niestandardowym progu decyzyjnym i prostą analizę kosztowo-korzyściową

### 📚 Dalsze materiały
- [Spark ML Tuning Guide](https://spark.apache.org/docs/latest/ml-tuning.html)
- [pyspark.ml.classification.GBTClassifier](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.GBTClassifier.html)

### 🔮 Co dalej?
Na **Wykładzie 9** przechodzimy do **Cloud Computing i GCP** — BigQuery i Dataproc! ☁️